#### stream (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:< rust_operators

:| ^ stream t =
    | StreamCons : t * (() -> stream t)
    | StreamNil

:= fold fn init s =
    := ^ body acc = \|
        | StreamCons (st, fn') => 루프 (fn acc st) (fn' ())
        | StreamNil => acc
    :& := 루프 acc = join_body body acc
    루프 init s

:= fold_back fn s init =
    := ^ body acc = \|
        | StreamCons (st, fn') => fn st (루프 acc (fn' ()))
        | StreamNil => acc
    :& := 루프 acc = join_body body acc
    루프 init s

:= to_list s =
    (s, [])
    ||> fold_back \ x acc =>
        x :: acc

:= rev s =
    (StreamNil, s)
    ||> fold \ s x =>
        StreamCons (x, \ () => s)

:= from_list list =
    (list, StreamNil)
    ||> listm::foldBack \ x acc =>
        StreamCons (x, \ () => acc)

#### from_list (test)

In [1]:
///- --test --symbol

listm::init 3i32 id
|> from_list
|> rev
|> to_list
|> _assert_eq [ 2; 1; 0 ]

__assert_eq { expected = Cons(2, Cons(1, Cons(0, Nil))) }

In [ ]:
:= try_item i s =
    := ^ body \* a {number} b. (i : a) : stream b -> option b = \|
        | StreamCons (x, _) ?& i <= 0 => Some x
        | StreamCons (_, fn) => 루프 (i - 1) (fn ())
        | StreamNil => None
    :& := 루프 \* a {number} b. (acc : a) (s' : stream b) : option b =
        ?= var_is acc, var_is s' ?|
        | !-, !- => body acc s'
        | _ =>
            := acc = dyn acc
            := s' = dyn s'
            ~~ body acc s'
    루프 i s

:= item i =
    try_item i >> optionm::value

#### try_item (test)

In [2]:
///- --test --symbol

listm::init 10i32 id
|> from_list
|> item 9i32
|> _assert_eq 9

__assert_eq { expected = 9 }

In [ ]:
:= new_infinite_stream fn =
    := ^ 루프 n =
        StreamCons (fn n, \ () => 루프 (n + 1))
    루프 0

:= new_infinite_stream_ fn =
    :~ ^ 루프 n =
        StreamCons (fn n, \ () => 루프 (n + 1))
    루프 0

#### new_infinite_stream (test)

In [3]:
///- --test --symbol

new_infinite_stream print_and_return
|> item 4i32
|> _assert_eq 4i32

print_and_return{ x = 0 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 3 }
print_and_return{ x = 4 }
__assert_eq { expected = 4 }

In [ ]:
:= new_finite_stream fn max =
    := ^ 루프 n =
        ? n >= max
        ?> StreamNil
        |: StreamCons (fn n, \ () => 루프 (n + 1))
    루프 0

:| memoized_stream t =
    | NotComputed : () -> stream t
    | Computed : stream t

:= memoize s =
    := ^ body \* t. (s : () -> stream t) : () -> stream t =
        := state = mut (NotComputed s)
        \ () =>
            ?= *state ?|
            | Computed x => x
            | NotComputed fn =>
                := new_state =
                    ?= fn () ?|
                    | StreamNil => StreamNil
                    | StreamCons (x, fn) => StreamCons (x, 루프 fn)
                state <- Computed new_state
                new_state
    :& := 루프 \* t. (s' : () -> stream t) : () -> stream t =
        join_body_unit body s s'
    루프 \ () => s

#### memoize (test)

In [4]:
///- --test --symbol

:= memo_stream = new_finite_stream print_and_return 10 |> memoize

memo_stream ()
|> item 3i32
|> _assert_eq 3i32

memo_stream ()
|> item 5i32
|> _assert_eq 5i32

print_and_return{ x = 0 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 3 }
__assert_eq { expected = 3 }
print_and_return{ x = 4 }
print_and_return{ x = 5 }
__assert_eq { expected = 5 }

In [5]:
///- --test --symbol

:= memo_stream = new_infinite_stream_ print_and_return |> memoize

memo_stream ()
|> item 3i32
|> _assert_eq 3i32

memo_stream ()
|> item 5i32
|> _assert_eq 5i32

print_and_return{ x = 0 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 3 }
__assert_eq { expected = 3 }
print_and_return{ x = 4 }
print_and_return{ x = 5 }
__assert_eq { expected = 5 }

In [ ]:
:= unfold f x0 =
    := ^ body x =
        ?= f x ?|
        | Some (x', y) => StreamCons (x', \ () => 루프 y)
        | None => StreamNil
    :& := 루프 x = join_body_unit body x0 x
    루프 x0

:= iterate f =
    \ x => Some (x, f x)
    |> unfold

#### iterate (test)

In [6]:
///- --test --symbol

iterate ((*) 2) 1i32
|> item 10i32
|> _assert_eq 1024

__assert_eq { expected = 1024 }

In [ ]:
:= iterate_map f m =
    \ x =>
        m x
        |> optionm::map \ x =>
            x, f x
    |> unfold

In [7]:
///- --test --symbol

iterate_map ((*) 2) Some 1i32
|> item 10i32
|> _assert_eq 1024

__assert_eq { expected = 1024 }

In [ ]:
:= take_while cond s =
    := ^ body i = \|
        | StreamCons (st, fn) ?& cond st i => StreamCons (st, \ () => 루프 (i + 1) (fn ()))
        | _ => StreamNil
    :& := 루프 i = join_body body i
    루프 0 s

:= sum seq =
    seq |> fold (+) 0

#### sum (test)

In [8]:
///- --test --symbol

listm::init 10i32 id
|> from_list
|> sum
|> _assert_eq 45

__assert_eq { expected = 45 }

In [9]:
///- --test --symbol

new_finite_stream print_and_return 10i32
|> take_while \ n (_ : i32) => n < 5
|> sum
|> _assert_eq 10

print_and_return{ x = 0 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 3 }
print_and_return{ x = 4 }
print_and_return{ x = 5 }
__assert_eq { expected = 10 }

In [10]:
///- --test --symbol

new_infinite_stream print_and_return
|> take_while \ n (_ : i32) => n < 5i32
|> sum
|> _assert_eq 10

print_and_return{ x = 0 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 3 }
print_and_return{ x = 4 }
print_and_return{ x = 5 }
__assert_eq { expected = 10 }

In [11]:
///- --test --symbol

iterate ((*) 6) 1i32
|> take_while \ _ i => i <= 7i32
|> to_list
|> _assert_eq [ 1i32; 6; 36; 216; 1296; 7776; 46656; 279936 ]

__assert_eq { expected = Cons(1, Cons(6, Cons(36, Cons(216, Cons(1296, Cons(7776, Cons(46656, Cons(279936, Nil)))))))) }

In [ ]:
:= indexed s =
    ((StreamNil, 0), s)
    ||> fold \ (acc, i) x =>
        StreamCons ((i, x), \ () => acc), i + 1
    |> fst
    |> rev

#### indexed (test)

In [12]:
///- --test --symbol

listm::init 10i32 ((*) 2)
|> from_list
|> indexed
|> item 5i32
|> _assert_eq (5i32, 10i32)

__assert_eq { expected = (5, 10) }

In [ ]:
:= map fn s =
    (s, StreamNil)
    ||> fold_back \ x acc =>
        StreamCons (fn x, \ () => acc)

#### map (test)

In [13]:
///- --test --symbol

listm::init 10i32 id
|> from_list
|> map ((*) 2)
|> item 5i32
|> _assert_eq 10i32

__assert_eq { expected = 10 }

In [ ]:
:= zip_with fn s1 s2 =
    := ^ 루프 s1 s2 =
        ?= s1, s2 ?|
        | StreamCons (st1, fn1), StreamCons (st2, fn2) =>
            StreamCons (fn st1 st2, \ () => 루프 (fn1 ()) (fn2 ()))
        | StreamNil, _ | _, StreamNil => StreamNil
    루프 s1 s2

:= zip_with_ fn s1 s2 =
    :~ ^ 루프 s1 s2 =
        ?= s1, s2 ?|
        | StreamCons (st1, fn1), StreamCons (st2, fn2) =>
            StreamCons (fn st1 st2, \ () => 루프 (fn1 ()) (fn2 ()))
        | StreamNil, _ | _, StreamNil => StreamNil
    루프 s1 s2

#### zip_with (test)

In [14]:
///- --test --symbol

((listm::init 10i32 id |> from_list), (listm::init 10i32 ((*) 2) |> from_list))
||> zip_with (+)
|> item 2i32
|> _assert_eq 6

__assert_eq { expected = 6 }

In [ ]:
:= zip s1 s2 =
    zip_with pair s1 s2

:= zip_ s1 s2 =
    zip_with_ pair s1 s2

#### zip (test)

In [15]:
///- --test --symbol

((listm::init 10i32 id |> from_list), (listm::init 10i32 ((*) 2) |> from_list))
||> zip
|> item 5i32
|> _assert_eq (5, 10)

__assert_eq { expected = (5, 10) }

In [ ]:
:= unzip s =
    := ^ body \* a b. (s : stream (a * b)) : stream a * stream b =
        ?= s ?|
        | StreamCons ((x, y), fn) =>
            := xs, ys = 루프 (fn ())
            StreamCons (x, \ () => xs), StreamCons (y, \ () => ys)
        | StreamNil => pair StreamNil StreamNil
    :& := 루프 \* a b. (x : stream (a * b)) : stream a * stream b =
        ? var_is x |> not
        ?> body x
        |:
            := x = dyn x
            ~~ body x
    루프 s

#### unzip (test)

In [16]:
///- --test --symbol

listm::init 10i32 id
|> listm::map \ x => x, x
|> from_list
|> unzip
|> \ x, y =>
    x |> sum
    |> _assert_eq 45

    y |> sum
    |> _assert_eq 45

__assert_eq { expected = 45 }
__assert_eq { expected = 45 }

In [17]:
:! io_error =
    `(
        rust::rust_type
            `($'std_io_Error')
            `($'std::io::Error')
            "type std_io_Error = string"
    )

:= new_io_error (text : string) : io_error =
    run_target_args (\ () => text) \|
        | Rust => \ text =>
            !\\(text, $'"std::io::Error::new(std::io::ErrorKind::Other, &*$0)"')
        | _ => \ text => text |> convert::unbox

:! buf_reader t =
    `(
        rust::rust_type
            `($'std_io_BufReader<`t>')
            `($'std::io::BufReader<`t>')
            "type std_io_BufReader<'T> = class end"
    )

:! cursor t =
    `(
        rust::rust_type
            `($'std_io_Cursor<`t>')
            `($'std::io::Cursor<`t>')
            "type std_io_Cursor<'T> = class end"
    )

:! buf_reader_tokio t =
    `(
        rust::rust_type
            `($'tokio_io_BufReader<`t>')
            `($'tokio::io::BufReader<`t>')
            "type tokio_io_BufReader<'T> = class end"
    )

:= new_buf_reader \* t. (x : t) : buf_reader t =
    !\\(x, $'"std::io::BufReader::new($0)"')

:= new_cursor \* t. (x : t) : cursor t =
    !\($'"std::io::Cursor::new(!x)"')

:! lines t =
    `(
        rust::rust_type
            `($'std_io_Lines<`t>')
            `($'std::io::Lines<`t>')
            "type std_io_Lines<'T> = class end"
    )

:= buf_read_lines \* t. (buf_reader : buf_reader t) : lines (buf_reader t) =
    !\($'"std::io::BufRead::lines(!buf_reader)"')

:! decode_reader_bytes t u =
    `(
        rust::rust_type
            `($'encoding_rs_io_DecodeReaderBytes<`t, `u>')
            `($'encoding_rs_io::DecodeReaderBytes<`t, `u>')
            "type encoding_rs_io_DecodeReaderBytes<'T, 'U> = class end"
    )

:= decode_reader_bytes_build \* t. (x : t) : decode_reader_bytes t (am'::vec u8) =
    !\\(x, $'"encoding_rs_io::DecodeReaderBytesBuilder::new().utf8_passthru(true).build($0)"')

:= buf_reader_read \* el dim.
    (slice : am'::slice' el dim) (buf_reader : buf_reader el) : resultm::result' convert::unativeint io_error
    =
    (!\($'"true; let mut !slice = !slice"') : bool) |> ignore
    !\($'"std::io::Read::read(&mut !buf_reader, &mut !slice)"')

:= io_read_by_ref \* t. (lines : lines t) : lines t =
    !\\(lines, $'"std::io::Read::by_ref($0)"')